# IBMF vs numerical integration of the Lotka-Volterra equations
Outputs of `LV_ODE_countFP` (`--method ibmf` and `--method ode`) on the same graph and from the same initial conditions,
compared with the functions of `compare_ODE_IBMF.py`. Figure for Sec. 4 of the notes.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, "/home/david/VSCProjects/Ecology/processing_data/IBMF")
from compare_ODE_IBMF import read_dir, load_graph, match, stability

path_runs = "/run/media/david/Data/Research/Ecology/Results/IBMF/ODE_vs_IBMF"
path_out = "/run/media/david/Data/Research/Ecology/Results/Plots_counting_fixed_points/ode_vs_ibmf"
graph_exe = "/home/david/VSCProjects/Ecology/Debug/print_graph_IBMF.out"
os.makedirs(path_out, exist_ok=True)
# (label, directory prefix, N, mu, sigma, seed of the graph)
cases = [("0.000", "s0", 1024, 0.370, 0.0, 14), ("0.150", "s15", 1024, 0.390, 0.15, 2)]


def set_style():
    plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "stix", "axes.linewidth": 1.2})


def format_axis(ax, tick_fontsize=13):
    ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=tick_fontsize, length=3)
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=tick_fontsize, length=5)

In [ ]:
from matplotlib.ticker import MaxNLocator
# 2 x 3 figure: rows sigma = 0 and 0.15; columns phi_ODE vs phi_IBMF, P(m_bar) over the runs, stability
set_style()
color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
fig, axs = plt.subplots(2, 3, figsize=(15, 8.4), dpi=150)
labels = iter('abcdef')
for row, (slab, pre, N, mu, sigma, g) in enumerate(cases):
    I, O = read_dir(f"{path_runs}/{pre}_ibmf"), read_dir(f"{path_runs}/{pre}_ode")
    n_init = I['attempts']
    A = load_graph(graph_exe, N, mu, sigma, 0.0, g)
    head = rf"$\sigma={float(slab):.2f}$, $N={N}$, $\mu={mu:.3f}$"
    # (a, d) phi of the fixed points found by both methods
    ax = axs[row, 0]
    io = match(I['M'], O['M'])
    common = np.nonzero(io >= 0)[0]
    pI, pO = I['counts'][common] / n_init, O['counts'][io[common]] / n_init
    ax.loglog(pI, pO, 'o', color=color_cycle[0], ms=5, label=f"{len(common)} fixed points in common")
    lim = np.array([0.3 / n_init, 1.0])
    ax.plot(lim, lim, color='gray', ls=':', lw=1)
    ax.set_xlabel(r"$\phi_{\rm IBMF}$", fontsize=16); ax.set_ylabel(r"$\phi_{\rm ODE}$", fontsize=16)
    ax.set_title(f"({next(labels)}) " + head, fontsize=13)
    ax.legend(loc='upper left', fontsize=10, frameon=False)
    # (b, e) distribution over the runs of the mean abundance of the fixed point reached
    ax = axs[row, 1]
    mb = {}
    for name, D in (('ibmf', I), ('ode', O)):
        ok = D['log'][:, 7] >= 0
        mb[name] = D['M'][D['log'][ok, 7].astype(int)].mean(1)
    bins = np.linspace(min(mb['ibmf'].min(), mb['ode'].min()), max(mb['ibmf'].max(), mb['ode'].max()), 41)
    for name, lab, col in (('ibmf', 'IBMF', color_cycle[0]), ('ode', 'ODE', color_cycle[1])):
        ax.hist(mb[name], bins=bins, histtype='step', lw=1.6, density=True, color=col, label=lab)
    ax.set_xlabel(r"$\bar m$", fontsize=16); ax.set_ylabel(r"$P(\bar m)$", fontsize=16)
    ax.set_title(f"({next(labels)}) " + head, fontsize=13)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0), useMathText=True)
    ax.ticklabel_format(axis='x', useOffset=False)
    ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.legend(loc='upper left', fontsize=10, frameon=False)
    # (c, f) stability: largest field of an extinct species vs smallest real part of eig(D_S M_SS)
    ax = axs[row, 2]
    for name, D, lab, col, mk in (('ibmf', I, 'IBMF', color_cycle[0], 'o'), ('ode', O, 'ODE', color_cycle[1], 's')):
        s = np.array([stability(A, m, sigma == 0) for m in D['M']])
        ax.plot(s[:, 1], s[:, 0], mk, ms=5, mfc='none', color=col, label=lab)
    ax.axvline(0, color='gray', lw=0.8, ls=':'); ax.axhline(0, color='gray', lw=0.8, ls=':')
    ax.set_xlabel(r"$\Lambda_{\min}$", fontsize=16); ax.set_ylabel(r"$f^{\rm ext}_{\max}$", fontsize=16)
    ax.set_title(f"({next(labels)}) " + head, fontsize=13)
    ax.ticklabel_format(style='sci', scilimits=(0, 0), useMathText=True)
    ax.legend(loc='lower right', fontsize=10, frameon=False)
for ax in axs.flat:
    format_axis(ax)
fig.tight_layout(w_pad=1.5)
fig.savefig(f"{path_out}/fig_ode_vs_ibmf.png")
plt.show()